In [1]:
%load_ext autoreload
%autoreload 2

import yaml
import os
import random
import sys
from dotenv import load_dotenv


load_dotenv()
    
# Añadir src al path
sys.path.append(os.path.abspath(os.path.join('../src')))

from promptnoises import process_json, CustomConfig
from data_utils import *
from model_utils import *

In [2]:
PREDS_FILENAME = os.getenv("PREDS_FILENAME", "../data/dataset_preds.json")
MODEL_NAME = os.getenv("MODEL_NAME", "prometheus-eval/prometheus-7b-v2.0")
MODEL_FT_PATH = os.getenv("MODEL_FT_PATH", "../output/prometheus_finetuned")
PREDS_PROMPT_COL = os.getenv("PREDS_PROMPT_COL", "user_content")
ROBUSTNESS_SEED = os.getenv("ROBUSTNESS_SEED", 42)
ROBUSTNESS_DATA_FILENAME = os.getenv("ROBUSTNESS_DATA_FILENAME",  "../output/output_robutness.json")

# Pruebas de Robustez (Robustness)
En esta fase del hackathon simularemos escenarios realistas donde los usuarios cometen errores tipográficos (typos), gramaticales o insertan ruidos naturales.
Se cambian parámetros del prompt ajustando CustomConfig para luego probar la resiliencia del modelo.

In [3]:

# 1. Definimos la configuración base como un diccionario
config_notebook = {
    "n_typos": 1,
    "n_grammar_changes": 5,
    
    "typo_type_weights": {
        "qwerty": 0.55,
        "omission": 0.25,
        "abbr": 0.20,
        "space_remove": 0.10
    },
    
    "vowel_delete_bias": 0.9,
    "abbr_q_weight": 0.6,
    "abbr_pq_weight": 0.4,
    
    "grammar_rule_weights": {
        "habia_to_habian": 1.0,
        "hemos_to_habemos": 0.9,
        "homophones": 0.8,
        "porque": 0.9,
        "seseo_ceceo": 0.9,
        "preterite_s": 0.9,
        "drop_initial_h": 0.9,
        "swap_bv": 0.9
    },
    
    "remove_open_questions": True,
    "strip_accents": True,
    "remove_commas": True,
    "lowercase": True
}

custom_cfg = CustomConfig(**config_notebook)

# si se quiere guardar se descomenta
#with open(CONFIG_FILENAME, 'w', encoding='utf-8') as f:
#    yaml.dump(my_params, f, default_flow_style=False, allow_unicode=True)


In [4]:

process_json(
    input_path=PREDS_FILENAME, 
    output_path=ROBUSTNESS_DATA_FILENAME, 
    custom_cfg=custom_cfg,
    input_col = PREDS_PROMPT_COL
)



In [8]:
df = load_data(ROBUSTNESS_DATA_FILENAME)
df.head()

,prompt_original,prompt_typos,prompt_grammatical_errors,prompt_custom
0,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and ovjective judge assi...,you are a fair strict and ovjectjve judge assi...
1,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and ovjective judge assi...,you are a fair strict and ovjective judge assi...
2,"You are a fair, strict, and objective judge as...","You are a fair, strict, and objective judge as...",You are a fair strict and ovjective judge assi...,you are a fair strict and ovjective judge assi...


In [7]:
# volver a cargar modelo
model, tokenizer = load_lora_model(MODEL_NAME, MODEL_FT_PATH)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [9]:
#1. formatear para prompt
#2. predicción para batches
# 3. se evalúa por variación de prompt y se compara resultados